# Frames index

Reads the headers of all raw frames of the given nights of one telescope (all types: zeros, darks, flats, science,
...) and saves them, one file per night, as `~/data/ocm_frames/<tel>/<night>.parquet`: one row per frame with the full
path and the header cards in `KEYS`. The other notebooks (`calib_stack`, `bias_qc`, `dark_qc`) then read these files
instead of opening every raw file (~0.17 s per file over NFS).

Nights already in the index are skipped, unless `OVERWRITE = True`. The newest night may still be growing during the
night: re-run it with `OVERWRITE = True` (or delete its file) to get the frames taken later.

In [12]:
import os
import glob
import pandas as pd
from astropy.io import fits
from tqdm.notebook import tqdm

TEL = 'zb08'
N_START = 1200     # first (oldest) night, int(JD) - 2460000
N_STOP = 1340      # last (newest) night
OVERWRITE = False  # True: read the nights again even if they are already in the index

RAW_DIR = f'/data/fits/{TEL}/raw'
FRAMES_DIR = os.path.expanduser(f'~/data/ocm_frames/{TEL}')

KEYS = ['DATE-OBS', 'JD', 'IMAGETYP', 'OBSTYPE', 'OBJECT', 'FILTER', 'EXPTIME', 'LOOP', 'NLOOPS',
        'T-CAM', 'T-CAMSET', 'READ-MOD', 'GAIN-MOD', 'XBINNING', 'YBINNING', 'NAXIS1', 'NAXIS2',
        'ZERO_ERA', 'DARK_ERA', 'FLAT_ERA', 'TEST', 'DOME-SHT', 'MIRR-COV', 'ALT_TEL', 'AZ_TEL', 'INSTRUME']

## Read the headers

In [13]:
os.makedirs(FRAMES_DIR, exist_ok=True)
for night in tqdm(range(N_START, N_STOP + 1), desc='nights'):
    out = f'{FRAMES_DIR}/{night}.parquet'
    if os.path.exists(out) and not OVERWRITE:
        print(night, 'already in the index')
        continue
    files = sorted(glob.glob(f'{RAW_DIR}/{night}/*.fits'))
    if not files:
        print(night, 'no frames')
        continue

    rows = []
    for f in tqdm(files, desc=f'night {night}', leave=False):
        h = fits.getheader(f)
        row = {'file': f, 'night': night}
        for k in KEYS:
            v = h.get(k)
            row[k] = None if v == '' else v  # empty cards as None (a column must not mix '' with numbers)
        rows.append(row)

    frames = pd.DataFrame(rows)
    frames.to_parquet(out)
    print(night, len(frames), 'frames:', ', '.join(f'{n} {t}' for t, n in frames['IMAGETYP'].value_counts().items()))

nights:   0%|          | 0/141 [00:00<?, ?it/s]

night 1200:   0%|          | 0/489 [00:00<?, ?it/s]

1200 489 frames: 421 science, 46 flat, 22 zero


night 1201:   0%|          | 0/580 [00:00<?, ?it/s]

1201 580 frames: 451 science, 107 flat, 22 zero


night 1202:   0%|          | 0/279 [00:00<?, ?it/s]

... (207 outputs left out)

night 1338:   0%|          | 0/443 [00:00<?, ?it/s]

1338 443 frames: 371 science, 50 flat, 22 zero
1339 no frames


night 1340:   0%|          | 0/138 [00:00<?, ?it/s]

1340 138 frames: 80 science, 47 flat, 11 zero


## Read the index back

How other notebooks read it: all nights of a block, one table.

In [14]:
frames = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f'{FRAMES_DIR}/*.parquet'))], ignore_index=True)
print(len(frames), 'frames in', frames['night'].nunique(), 'nights')
frames.groupby(['night', 'IMAGETYP']).size().unstack(fill_value=0)

83055 frames in 108 nights


IMAGETYP,dark,flat,science,zero
night,,,,
1200,0,46,421,22
1201,0,107,451,22
1202,6,56,164,53
1203,0,108,1231,22
1204,6,56,116,53
...,...,...,...,...
1332,18,0,0,76
1334,0,0,104,12
1335,0,0,15,11
